# Scene Detection & Face-Segment Extraction

Uses **TransNetV2** for shot boundaries and **dlib** for face detection.
Configure paths in the cells below before running.


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import dlib


In [ ]:
# Clone TransNetV2 if not already present (run once from your project root)
# !git clone https://github.com/soCzech/TransNetV2.git


In [ ]:
# Change into the TransNetV2 directory after cloning
# %cd TransNetV2


In [ ]:
# Optional: list TransNetV2 repository contents
# !ls


In [ ]:
# Change into the inference subdirectory
# %cd inference


In [ ]:
# Optional: list inference directory contents
# !ls


In [ ]:
# Requires TransNetV2/inference on your Python path (see clone/%cd cells above)
from transnetv2 import TransNetV2

In [ ]:
# Loads pretrained weights from TransNetV2/inference/transnetv2-weights/
model = TransNetV2()

In [ ]:
# --- Environment setup (configure before running) ---
# Google Colab only — uncomment to mount Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# --- Paths (update these for your environment) ---
# Input: directory containing raw .mp4 / .avi / .mov / .mkv files
VIDEO_DIR = 'data/raw_videos'


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install tensorflow
# !pip install imageio[ffmpeg]


In [ ]:
# !pip install ffmpeg-python pillow


In [ ]:
# Verify ffmpeg is available on your system
# !ffmpeg -version


In [ ]:
# Linux/Colab only — install ffmpeg via apt (uncomment if needed)
# !apt-get update
# !apt-get install -y ffmpeg


In [ ]:
# Optional: list your data directory to verify paths
# !ls data/raw_videos


In [ ]:
# Optional: inspect a condition-specific folder
# !ls data/raw_videos/drunk


## Batch processing

Set `VIDEO_DIR` and `main_output_dir` above, then run the cells below.


In [ ]:
# Output: directory where face-containing scene segments will be saved
main_output_dir = 'data/segments'
os.makedirs(main_output_dir, exist_ok=True)


In [ ]:
# Load dlib's face detector
face_detector = dlib.get_frontal_face_detector()

In [ ]:
# Get a list of all video files in the specified directory
video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.avi', '.mov', '.mkv'))]


In [ ]:
# Iterate through each video file in the directory
for video_file in video_files:
    # Full path to the current video file
    video_path = os.path.join(VIDEO_DIR, video_file)

    # Create a subdirectory for this video in the output directory
    video_name = os.path.splitext(video_file)[0]  # Get the video name without extension
    video_output_dir = os.path.join(main_output_dir, video_name)
    os.makedirs(video_output_dir, exist_ok=True)

    # Open the video file using OpenCV
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_size = (width, height)

    # Load the video and predict scenes using TransNetV2
    video_frames, single_frame_predictions, all_frame_predictions = model.predict_video(video_path)

    # Get the scenes as frame indices (start, end) from predictions
    scenes = model.predictions_to_scenes(single_frame_predictions)

    # Threshold for saving segments (e.g., stop saving if no faces for 5 frames)
    max_no_face_frames = 5

    # Iterate through the scenes, extract each scene and check for faces
    for i, (start_frame, end_frame) in enumerate(scenes):
        # Set the video capture to the start frame of the current scene
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        face_found = False
        consecutive_no_face_frames = 0
        saving_segment = False
        segment_count = 0
        out = None

        # Iterate through each frame in the scene
        for frame_num in range(start_frame, end_frame + 1):
            ret, frame = cap.read()
            if not ret:
                break

            # Convert the frame to grayscale for face detection
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Detect faces in the frame using dlib
            faces = face_detector(gray_frame)

            if len(faces) > 0:
                # If faces are detected, start or continue saving the segment
                if not saving_segment:
                    # Create an output filename for the scene segment
                    segment_count += 1
                    output_path = os.path.join(video_output_dir, f'segment_with_face_{i+1}_{segment_count}.mp4')

                    # Open a video writer for the scene segment
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    out = cv2.VideoWriter(output_path, fourcc, fps, frame_size)
                    saving_segment = True

                # Write the frame to the current segment
                out.write(frame)

                # Reset the no face counter
                consecutive_no_face_frames = 0

            else:
                # No face detected
                consecutive_no_face_frames += 1

                # If we're saving and hit the threshold for no faces, stop saving
                if saving_segment and consecutive_no_face_frames >= max_no_face_frames:
                    out.release()  # Release the video writer for this segment
                    saving_segment = False

        # If a segment is still being saved, release the writer
        if saving_segment:
            out.release()

    # Release the video capture for the current video file
    cap.release()

print(f'Successfully saved parts of the scenes with faces to {main_output_dir}')